# Análise Experimental de Desempenho e Escalabilidade

Este caderno interativo reúne e explora todos os resultados dos testes experimentais realizados sobre a arquitetura de Digital Twin baseada em Asset Administration Shells (AAS):

1. **Parte 1: Validação Experimental no Banco Piloto FMS**
   - Análise de execução individual do banco real FMS (5 conveyors + 1 punch station + sensores + atuadores).
   - Métricas de tempo de inicialização (Startup Time), fluidez (FPS) e latência de frame (Frame Time).
   - Evolução temporal ao longo da execução do ciclo industrial.
   - Distribuições acumuladas empíricas (CDF).

2. **Parte 2: Testes de Stress e Escalabilidade Paramétrica (5 a 1000 Conveyors)**
   - Avaliação da complexidade algorítmica ($O(N)$) do processo de ingestão e geração procedural.
   - Degradação de fluidez em regime estacionário e percentis críticos (1% Low FPS).
   - Latência de frame, jitter e cauda de latência (P95 e P99).
   - Custos computacionais unitários por módulo de conveyor.
   - Dinâmica temporal ao longo de 60 segundos por patamar de carga.
   - Matriz de correlação linear de Pearson e Coeficiente de Variação (CV%).


In [ ]:
import os
import re
import glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Configuração de Estilo Visual
sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams.update({
    'font.size': 11,
    'axes.labelsize': 12,
    'axes.titlesize': 13,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'legend.fontsize': 10,
    'figure.titlesize': 14,
    'figure.dpi': 150
})
%matplotlib inline

# Diretório para salvar imagens exportadas
OUTPUT_DIR_IMG = "graficos_fms"
os.makedirs(OUTPUT_DIR_IMG, exist_ok=True)


---
# PARTE 1: Análise Individual do Banco Piloto FMS

Esta secção analisa o teste individual e dedicado realizado sobre a réplica digital do banco modular físico **FMS** (composto pela topologia real de 5 módulos de conveyor, 1 posto de furação/estampagem - punch mechanism, e instrumentação completa com sensores indutivos/magnéticos e atuadores).

Foram realizadas **100 execuções completas e independentes** do ciclo de vida da simulação para avaliar a estabilidade, determinismo de carregamento e fluidez de execução em condições reais de operação.


In [ ]:
def load_fms_data(fms_dir="FMS"):
    csv_files = glob.glob(os.path.join(fms_dir, "**", "*.csv"), recursive=True)
    if not csv_files:
        csv_files = glob.glob(os.path.join("AAS Client", "Resultados", "FMS", "**", "*.csv"), recursive=True)
    
    print(f"Ficheiros CSV do FMS encontrados: {len(csv_files)}")
    runs_data = []
    timeseries_data = []
    
    for idx, filepath in enumerate(csv_files):
        try:
            df = pd.read_csv(filepath)
            if df.empty or 'Timestamp_s' not in df.columns:
                continue
            
            startup_time = df['StartupTime_s'].max()
            if startup_time <= 0:
                completed = df[df['IsStartupComplete'] == 1]
                startup_time = completed['Timestamp_s'].iloc[0] if not completed.empty else np.nan
            
            df_steady = df[df['IsStartupComplete'] == 1]
            if df_steady.empty:
                df_steady = df[df['Timestamp_s'] > startup_time]
            if df_steady.empty:
                df_steady = df.tail(len(df) // 2)
            
            fps_mean = df_steady['FPS'].mean()
            fps_min = df_steady['FPS'].min()
            fps_p01 = df_steady['FPS'].quantile(0.01)
            
            ft_mean = df_steady['FrameTime_ms'].mean()
            ft_p95 = df_steady['FrameTime_ms'].quantile(0.95)
            ft_p99 = df_steady['FrameTime_ms'].quantile(0.99)
            ft_std = df_steady['FrameTime_ms'].std()
            
            duration_s = df['Timestamp_s'].max()
            
            runs_data.append({
                'Run': idx,
                'StartupTime_s': startup_time,
                'FPS_Mean': fps_mean,
                'FPS_Min': fps_min,
                'FPS_P01_Low': fps_p01,
                'FrameTime_Mean_ms': ft_mean,
                'FrameTime_P95_ms': ft_p95,
                'FrameTime_P99_ms': ft_p99,
                'FrameTime_Std_ms': ft_std,
                'Duration_s': duration_s
            })
            
            df_ts = df[['Timestamp_s', 'FPS', 'FrameTime_ms']].copy()
            df_ts['Time_bin'] = df_ts['Timestamp_s'].round(1)
            df_ts['Run'] = idx
            timeseries_data.append(df_ts)
        except Exception as e:
            continue
            
    return pd.DataFrame(runs_data), pd.concat(timeseries_data, ignore_index=True)

df_fms_runs, df_fms_ts = load_fms_data()
print(f"Total de execuções FMS carregadas com sucesso: {len(df_fms_runs)}")
df_fms_runs.head(3)


### 1.1 Tabela Resumo das Métricas do Banco FMS (Média $\pm$ Desvio Padrão)

In [ ]:
fms_stats = df_fms_runs.agg({
    'StartupTime_s': ['mean', 'std', 'min', 'max'],
    'FPS_Mean': ['mean', 'std', 'min', 'max'],
    'FPS_P01_Low': ['mean', 'std', 'min', 'max'],
    'FrameTime_Mean_ms': ['mean', 'std', 'min', 'max'],
    'FrameTime_P95_ms': ['mean', 'std', 'min', 'max'],
    'FrameTime_P99_ms': ['mean', 'std', 'min', 'max'],
    'Duration_s': ['mean', 'std', 'min', 'max']
}).round(2)
display(fms_stats)


### 1.2 Distribuição de Startup Time e Taxa de Atualização (FPS) no Banco FMS


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9, 4.2))

# Subplot 1: Startup Time
st_m = df_fms_runs['StartupTime_s'].mean()
st_s = df_fms_runs['StartupTime_s'].std()
sns.boxplot(y=df_fms_runs['StartupTime_s'], ax=axes[0], color='#2b5c8f', width=0.35, showmeans=True,
            meanprops={'marker':'o', 'markerfacecolor':'red', 'markeredgecolor':'black', 'markersize':8})
axes[0].set_title(f'Startup Initialization Time\n(Mean: {st_m:.2f}s ± {st_s:.2f}s)', pad=10)
axes[0].set_ylabel('Startup Time (seconds)')

# Subplot 2: FPS
fps_m = df_fms_runs['FPS_Mean'].mean()
fps_s = df_fms_runs['FPS_Mean'].std()
sns.boxplot(y=df_fms_runs['FPS_Mean'], ax=axes[1], color='#27ae60', width=0.35, showmeans=True,
            meanprops={'marker':'o', 'markerfacecolor':'red', 'markeredgecolor':'black', 'markersize':8})
axes[1].axhline(60, color='#c0392b', linestyle='--', linewidth=1.5, label='60 FPS Reference Target')
axes[1].set_title(f'Steady-State Rendering Rate\n(Mean: {fps_m:.1f} ± {fps_s:.1f} FPS)', pad=10)
axes[1].set_ylabel('Frames per Second (FPS)')
axes[1].legend(loc='lower right')

fig.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR_IMG, 'fms_metrics_distributions.png'), dpi=300, bbox_inches='tight')
fig.savefig(os.path.join(OUTPUT_DIR_IMG, 'fms_metrics_distributions.pdf'), dpi=300, bbox_inches='tight')
plt.show()


### 1.3 Evolução Temporal Dinâmica do Banco FMS (FPS e Latência)

In [ ]:
ts_agg = df_fms_ts.groupby('Time_bin').agg({
    'FPS': ['mean', 'std'],
    'FrameTime_ms': ['mean', 'std']
}).reset_index()
ts_agg.columns = ['Time_bin', 'FPS_mean', 'FPS_std', 'FT_mean', 'FT_std']
ts_agg = ts_agg[ts_agg['Time_bin'] <= 7.5]

fig, ax1 = plt.subplots(figsize=(8.5, 4.5))
color = '#27ae60'
ax1.set_xlabel('Tempo Decorrido (s)')
ax1.set_ylabel('FPS Médio', color=color)
ax1.plot(ts_agg['Time_bin'], ts_agg['FPS_mean'], color=color, linewidth=2, label='FPS Médio')
ax1.fill_between(ts_agg['Time_bin'], ts_agg['FPS_mean'] - ts_agg['FPS_std'], ts_agg['FPS_mean'] + ts_agg['FPS_std'], color=color, alpha=0.2)
ax1.tick_params(axis='y', labelcolor=color)
ax1.axvline(st_m, color='#2980b9', linestyle=':', linewidth=2, label=f'Fim do Startup (~{st_m:.2f}s)')

ax2 = ax1.twinx()
color = '#e67e22'
ax2.set_ylabel('Latência de Frame (ms)', color=color)
ax2.plot(ts_agg['Time_bin'], ts_agg['FT_mean'], color=color, linewidth=2, linestyle='--', label='Frame Time (ms)')
ax2.tick_params(axis='y', labelcolor=color)

fig.suptitle('Evolução Dinâmica da Execução do Piloto FMS', y=1.02)
fig.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR_IMG, "fms_timeseries_evolution.png"), dpi=300, bbox_inches='tight')
fig.savefig(os.path.join(OUTPUT_DIR_IMG, "fms_timeseries_evolution.pdf"), dpi=300, bbox_inches='tight')
plt.show()


### 1.4 Distribuição Cumulativa Empírica (CDF) do Banco FMS

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

# CDF Frame Time
vals_ft = np.sort(df_fms_runs['FrameTime_Mean_ms'].dropna())
y_ft = np.arange(1, len(vals_ft) + 1) / len(vals_ft)
axes[0].step(vals_ft, y_ft, color='#e67e22', linewidth=2.2, label='FMS Bench Frame Time')
axes[0].axvline(16.67, color='green', linestyle=':', label='Meta 60 FPS (16.7 ms)')
axes[0].set_title('CDF Empírica da Latência Média de Frame')
axes[0].set_xlabel('Latência Média de Frame (ms)')
axes[0].set_ylabel('Probabilidade Acumulada P(X ≤ x)')
axes[0].legend()

# CDF Startup Time
vals_st = np.sort(df_fms_runs['StartupTime_s'].dropna())
y_st = np.arange(1, len(vals_st) + 1) / len(vals_st)
axes[1].step(vals_st, y_st, color='#2b5c8f', linewidth=2.2, label='FMS Startup Time')
axes[1].set_title('CDF Empírica do Tempo de Inicialização')
axes[1].set_xlabel('Tempo de Inicialização (s)')
axes[1].set_ylabel('Probabilidade Acumulada P(X ≤ x)')
axes[1].legend()

fig.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR_IMG, "fms_cdfs.png"), dpi=300, bbox_inches='tight')
fig.savefig(os.path.join(OUTPUT_DIR_IMG, "fms_cdfs.pdf"), dpi=300, bbox_inches='tight')
plt.show()


---
# PARTE 2: Testes de Stress e Escalabilidade Paramétrica (5 a 1000 Conveyors)

Nesta segunda parte, avalia-se o comportamento do motor de simulação e da pipeline AAS sob condições extremas de carga e densidade de ativos, escalando de **5 até 1000 conveyors** com postos de furação periódicos.

São processadas **800 execuções automatizadas** (100 por cada patamar de escala: 5, 10, 25, 50, 100, 250, 500 e 1000 conveyors).


In [ ]:
def extract_conveyor_count(folder_name):
    match = re.search(r'(\d+)', folder_name)
    return int(match.group(1)) if match else None

def load_stress_metrics(base_dir="."):
    subdirs = [d for d in os.listdir(base_dir) if os.path.isdir(os.path.join(base_dir, d))]
    conveyor_folders = []
    for d in subdirs:
        if d.lower().startswith(('.', 'aasx', 'output', 'graficos', 'charts', 'fms')):
            continue
        count = extract_conveyor_count(d)
        if count is not None:
            conveyor_folders.append((count, d))
    conveyor_folders.sort(key=lambda x: x[0])

    runs_data = []
    timeseries_data = []

    for count, folder_name in conveyor_folders:
        folder_path = os.path.join(base_dir, folder_name)
        csv_files = glob.glob(os.path.join(folder_path, "**", "*.csv"), recursive=True)
        if count == 5 and len(csv_files) > 100:
            csv_files = sorted(csv_files, key=os.path.getmtime, reverse=True)[:100]

        print(f"A carregar '{folder_name}' ({count} conveyors) - {len(csv_files)} execuções...")

        for run_idx, filepath in enumerate(csv_files):
            try:
                df = pd.read_csv(filepath)
                if df.empty or 'Timestamp_s' not in df.columns:
                    continue

                startup_time = df['StartupTime_s'].max()
                if startup_time <= 0:
                    completed = df[df['IsStartupComplete'] == 1]
                    startup_time = completed['Timestamp_s'].iloc[0] if not completed.empty else np.nan

                df_startup = df[df['IsStartupComplete'] == 0]
                df_steady = df[df['IsStartupComplete'] == 1]
                if df_steady.empty:
                    df_steady = df

                fps_mean = df_steady['FPS'].mean()
                fps_min = df_steady['FPS'].min()
                fps_p01 = df_steady['FPS'].quantile(0.01)
                fps_startup = df_startup['FPS'].mean() if not df_startup.empty else np.nan

                frametime_mean = df_steady['FrameTime_ms'].mean()
                frametime_p95 = df_steady['FrameTime_ms'].quantile(0.95)
                frametime_p99 = df_steady['FrameTime_ms'].quantile(0.99)
                frametime_std = df_steady['FrameTime_ms'].std()

                pct_below_60 = (df_steady['FPS'] < 59.0).mean() * 100.0
                pct_below_30 = (df_steady['FPS'] < 29.0).mean() * 100.0

                runs_data.append({
                    'Conveyors': count,
                    'Run': run_idx,
                    'StartupTime_s': startup_time,
                    'FPS_Mean': fps_mean,
                    'FPS_Min': fps_min,
                    'FPS_P01_Low': fps_p01,
                    'FPS_Startup_Mean': fps_startup,
                    'FrameTime_Mean_ms': frametime_mean,
                    'FrameTime_P95_ms': frametime_p95,
                    'FrameTime_P99_ms': frametime_p99,
                    'FrameTime_Std_ms': frametime_std,
                    'Pct_Frames_Below_60': pct_below_60,
                    'Pct_Frames_Below_30': pct_below_30,
                    'Startup_per_Conveyor_ms': (startup_time / count) * 1000.0,
                    'FrameTime_per_Conveyor_us': (frametime_mean / count) * 1000.0,
                })

                df_ts = df[['Timestamp_s', 'FPS', 'FrameTime_ms']].copy()
                df_ts['Time_bin'] = df_ts['Timestamp_s'].round(0)
                df_ts['Conveyors'] = count
                timeseries_data.append(df_ts)

            except Exception as e:
                continue

    return pd.DataFrame(runs_data), pd.concat(timeseries_data, ignore_index=True)

df_runs, df_ts = load_stress_metrics(".")
print(f"Total de execuções de stress carregadas: {len(df_runs)}")
df_runs.head(3)


### Tabela Resumo Consolidada do Teste de Stress (Médias $\pm$ Desvios Padrão)

In [ ]:
stats = df_runs.groupby('Conveyors').agg({
    'StartupTime_s': ['mean', 'std'],
    'FPS_Mean': ['mean', 'std'],
    'FrameTime_Mean_ms': ['mean', 'std'],
}).round(2)
display(stats)


## 2. Análise do Tempo de Inicialização (Startup Time)

In [ ]:
# 2.1 Startup Time Boxplot
plt.figure(figsize=(8, 4.5))
sns.boxplot(data=df_runs, x='Conveyors', y='StartupTime_s', hue='Conveyors', legend=False, palette='Blues_d', showmeans=True,
            meanprops={"marker":"o", "markerfacecolor":"red", "markeredgecolor":"black", "markersize":6})
plt.title("Startup Time Distribution by Scale")
plt.xlabel("Number of Conveyors")
plt.ylabel("Startup Time (s)")
plt.show()

# 2.2 Startup Time Violin Plot
plt.figure(figsize=(8, 4.5))
sns.violinplot(data=df_runs, x='Conveyors', y='StartupTime_s', hue='Conveyors', legend=False, palette='Blues', inner="quartile")
plt.title("Startup Time Density and Quartiles")
plt.xlabel("Number of Conveyors")
plt.ylabel("Startup Time (s)")
plt.show()

# 2.3 Trend Curve with Standard Deviation
startup_agg = df_runs.groupby('Conveyors')['StartupTime_s'].agg(['mean', 'std']).reset_index()
plt.figure(figsize=(8, 4.5))
plt.plot(startup_agg['Conveyors'], startup_agg['mean'], marker='o', color='#1f77b4', linewidth=2.2, label='Mean')
plt.fill_between(startup_agg['Conveyors'], 
                 startup_agg['mean'] - startup_agg['std'], 
                 startup_agg['mean'] + startup_agg['std'], 
                 alpha=0.25, color='#1f77b4', label='±1 Standard Deviation')
plt.title("Mean Startup Time vs. Number of Conveyors")
plt.xlabel("Number of Conveyors")
plt.ylabel("Startup Time (s)")
plt.legend()
plt.show()

# 2.4 Log-Log Scale Analysis (Algorithmic Complexity)
plt.figure(figsize=(8, 4.5))
plt.plot(startup_agg['Conveyors'], startup_agg['mean'], marker='s', color='#2b5c8f', linewidth=2, label='Experimental Mean')
plt.xscale('log')
plt.yscale('log')
plt.title("Startup Time in Log-Log Scale (Complexity Analysis)")
plt.xlabel("Number of Conveyors (log scale)")
plt.ylabel("Startup Time (s, log scale)")
plt.legend()
plt.show()

# 2.5 Unit Startup Cost per Conveyor
plt.figure(figsize=(8, 4.5))
sns.barplot(data=df_runs, x='Conveyors', y='Startup_per_Conveyor_ms', hue='Conveyors', legend=False, palette='mako', errorbar='sd', capsize=0.1)
plt.title("Mean Unit Startup Time per Conveyor (ms/conveyor)")
plt.xlabel("Number of Conveyors")
plt.ylabel("Instantiation Cost (ms/conveyor)")
plt.show()


## 3. Fluidez de Renderização e Simulação (FPS)

In [ ]:
# 3.1 Mean FPS Boxplot
plt.figure(figsize=(8, 4.5))
sns.boxplot(data=df_runs, x='Conveyors', y='FPS_Mean', hue='Conveyors', legend=False, palette='Greens_d', showmeans=True,
            meanprops={"marker":"o", "markerfacecolor":"red", "markeredgecolor":"black", "markersize":6})
plt.axhline(60, color='r', linestyle='--', label='Target 60 FPS')
plt.axhline(30, color='gray', linestyle=':', label='Threshold 30 FPS')
plt.title("Mean FPS")
plt.xlabel("Number of Conveyors")
plt.ylabel("Frames per Second (FPS)")
plt.legend()
plt.show()

# 3.2 Mean FPS Degradation (Curve with Standard Deviation)
fps_agg = df_runs.groupby('Conveyors')['FPS_Mean'].agg(['mean', 'std']).reset_index()
plt.figure(figsize=(8, 4.5))
plt.plot(fps_agg['Conveyors'], fps_agg['mean'], marker='o', color='#2ca02c', linewidth=2.2, label='Mean FPS')
plt.fill_between(fps_agg['Conveyors'], 
                 fps_agg['mean'] - fps_agg['std'], 
                 fps_agg['mean'] + fps_agg['std'], 
                 alpha=0.25, color='#2ca02c', label='±1 Standard Deviation')
plt.axhline(60, color='r', linestyle='--', label='Target 60 FPS')
plt.axhline(30, color='gray', linestyle=':', label='Threshold 30 FPS')
plt.title("Mean FPS Degradation with Increasing Scale")
plt.xlabel("Number of Conveyors")
plt.ylabel("Frames per Second (FPS)")
plt.legend()
plt.show()

# 3.3 Smoothness Stability: Mean FPS vs 1% Low FPS
fps_low_agg = df_runs.groupby('Conveyors')['FPS_P01_Low'].agg(['mean']).reset_index()
plt.figure(figsize=(8, 4.5))
plt.plot(fps_agg['Conveyors'], fps_agg['mean'], marker='o', color='#2ca02c', linewidth=2, label='Mean Steady-State FPS')
plt.plot(fps_low_agg['Conveyors'], fps_low_agg['mean'], marker='v', color='#d62728', linewidth=2, linestyle='--', label='1% Low FPS')
plt.axhline(60, color='gray', linestyle=':', alpha=0.7, label='60 FPS Target')
plt.title("Smoothness Stability: Mean FPS vs. 1% Low FPS")
plt.xlabel("Number of Conveyors")
plt.ylabel("Frames per Second (FPS)")
plt.legend()
plt.show()

# 3.4 Comparison: Startup Phase FPS vs Steady State
phases_df = df_runs.groupby('Conveyors')[['FPS_Startup_Mean', 'FPS_Mean']].mean().reset_index()
phases_melted = phases_df.melt(id_vars='Conveyors', var_name='Phase', value_name='FPS')
phases_melted['Phase'] = phases_melted['Phase'].map({'FPS_Startup_Mean': 'Startup Phase', 'FPS_Mean': 'Steady State'})
plt.figure(figsize=(8, 4.5))
sns.barplot(data=phases_melted, x='Conveyors', y='FPS', hue='Phase', palette='Set1')
plt.axhline(60, color='black', linestyle=':', alpha=0.6)
plt.title("FPS Comparison: Startup vs Steady State")
plt.xlabel("Number of Conveyors")
plt.ylabel("FPS")
plt.legend()
plt.show()

# 3.5 Percentage of Frames with Fluidity Loss (<60 FPS and <30 FPS)
pct_melted = df_runs.melt(id_vars=['Conveyors'], value_vars=['Pct_Frames_Below_60', 'Pct_Frames_Below_30'],
                          var_name='Threshold', value_name='Percentage')
pct_melted['Threshold'] = pct_melted['Threshold'].map({
    'Pct_Frames_Below_60': '< 60 FPS',
    'Pct_Frames_Below_30': '< 30 FPS'
})
plt.figure(figsize=(8, 4.5))
sns.barplot(data=pct_melted, x='Conveyors', y='Percentage', hue='Threshold', palette='Reds', errorbar=None)
plt.title("Percentage of Frames Below Critical Thresholds")
plt.xlabel("Number of Conveyors")
plt.ylabel("% of Affected Frames")
plt.legend()
plt.show()


## 4. Latência de Renderização (Frame Time em Milissegundos)

In [ ]:
# 4.1 Mean Frame Time Boxplot
plt.figure(figsize=(8, 4.5))
sns.boxplot(data=df_runs, x='Conveyors', y='FrameTime_Mean_ms', hue='Conveyors', legend=False, palette='Oranges_d', showmeans=True,
            meanprops={"marker":"o", "markerfacecolor":"red", "markeredgecolor":"black", "markersize":6})
plt.axhline(16.67, color='r', linestyle='--', label='60 FPS Limit (16.7 ms)')
plt.axhline(33.33, color='gray', linestyle=':', label='30 FPS Limit (33.3 ms)')
plt.title("Mean Frame Time")
plt.xlabel("Number of Conveyors")
plt.ylabel("Frame Time (ms)")
plt.legend()
plt.show()

# 4.2 Tail Latency: P95 and P99 Percentiles
lat_agg = df_runs.groupby('Conveyors').agg({
    'FrameTime_Mean_ms': 'mean',
    'FrameTime_P95_ms': 'mean',
    'FrameTime_P99_ms': 'mean'
}).reset_index()
plt.figure(figsize=(8, 4.5))
plt.plot(lat_agg['Conveyors'], lat_agg['FrameTime_Mean_ms'], marker='o', color='#2b5c8f', linewidth=2, label='Mean')
plt.plot(lat_agg['Conveyors'], lat_agg['FrameTime_P95_ms'], marker='s', color='#e67e22', linewidth=2, label='P95')
plt.plot(lat_agg['Conveyors'], lat_agg['FrameTime_P99_ms'], marker='^', color='#c0392b', linewidth=2, linestyle='--', label='P99')
plt.axhline(16.67, color='green', linestyle=':', label='16.7 ms (60 FPS)')
plt.title("Latency Scaling: Mean vs P95 vs P99")
plt.xlabel("Number of Conveyors")
plt.ylabel("Latency (ms)")
plt.legend()
plt.show()

# 4.3 Frame Time Jitter (Standard Deviation per Run)
plt.figure(figsize=(8, 4.5))
sns.boxplot(data=df_runs, x='Conveyors', y='FrameTime_Std_ms', hue='Conveyors', legend=False, palette='magma')
plt.title("Frame Time Jitter (Frame Pacing Instability)")
plt.xlabel("Number of Conveyors")
plt.ylabel("Frame Time Standard Deviation (ms)")
plt.show()

# 4.4 Unit Computational Frame Cost per Conveyor (µs/conveyor)
plt.figure(figsize=(8, 4.5))
sns.barplot(data=df_runs, x='Conveyors', y='FrameTime_per_Conveyor_us', hue='Conveyors', legend=False, palette='viridis', errorbar='sd', capsize=0.1)
plt.title("Unit Computational Frame Cost per Conveyor (µs/conveyor)")
plt.xlabel("Number of Conveyors")
plt.ylabel("Frame Cost (µs/conveyor)")
plt.show()


## 5. Evolução Temporal da Simulação ao Longo dos Segundos

In [ ]:
conveyors_order = sorted(df_runs['Conveyors'].unique())
colors = sns.color_palette("tab10", len(conveyors_order))
df_ts_filt = df_ts[df_ts['Time_bin'] <= 65]
ts_mean = df_ts_filt.groupby(['Conveyors', 'Time_bin'])[['FPS', 'FrameTime_ms']].mean().reset_index()

plt.figure(figsize=(10, 5))
for idx, c in enumerate(conveyors_order):
    sub = ts_mean[ts_mean['Conveyors'] == c]
    plt.plot(sub['Time_bin'], sub['FPS'], label=f"{c} conveyors", linewidth=1.8, color=colors[idx % len(colors)])
plt.axhline(60, color='black', linestyle=':', label='Target 60 FPS')
plt.title("Temporal Evolution of Mean FPS During Simulation")
plt.xlabel("Elapsed Time (s)")
plt.ylabel("Frames per Second (FPS)")
plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.show()


## 6. Análise Estatística Avançada e Correlações

In [ ]:
# 6.1 Empirical Cumulative Distribution Function (CDF) of Frame Time
plt.figure(figsize=(8, 4.5))
for idx, c in enumerate(conveyors_order):
    vals = np.sort(df_runs[df_runs['Conveyors'] == c]['FrameTime_Mean_ms'].dropna())
    if len(vals) > 0:
        y = np.arange(1, len(vals) + 1) / len(vals)
        plt.step(vals, y, label=f"{c} conveyors", linewidth=2, color=colors[idx % len(colors)])
plt.axvline(16.67, color='green', linestyle=':', label='16.7 ms (60 FPS)')
plt.axvline(33.33, color='red', linestyle=':', label='33.3 ms (30 FPS)')
plt.title("Cumulative Distribution Function (CDF) of Frame Time")
plt.xlabel("Mean Frame Time (ms)")
plt.ylabel("Cumulative Probability P(X ≤ x)")
plt.legend()
plt.show()

# 6.2 Pearson Linear Correlation Matrix
corr_cols = [
    'Conveyors', 'StartupTime_s', 'FPS_Mean', 'FPS_P01_Low',
    'FrameTime_Mean_ms', 'FrameTime_P95_ms'
]
corr_rename = {
    'Conveyors': 'Conveyors',
    'StartupTime_s': 'Startup Time',
    'FPS_Mean': 'Mean FPS',
    'FPS_P01_Low': '1% Low FPS',
    'FrameTime_Mean_ms': 'Mean Frame Time',
    'FrameTime_P95_ms': 'Frame Time P95'
}
corr_matrix = df_runs[corr_cols].rename(columns=corr_rename).corr()
plt.figure(figsize=(8, 6.5))
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap='coolwarm', vmin=-1, vmax=1, square=True)
plt.title('Correlation Matrix of Performance Metrics')
plt.tight_layout()
plt.show()

# 6.3 Coefficient of Variation (CV %): Consistency Verification across Runs
stats_cv = df_runs.groupby('Conveyors').agg({
    'StartupTime_s': lambda x: (x.std() / x.mean()) * 100.0,
    'FPS_Mean': lambda x: (x.std() / x.mean()) * 100.0,
    'FrameTime_Mean_ms': lambda x: (x.std() / x.mean()) * 100.0,
}).reset_index()

cv_melted = stats_cv.melt(id_vars=['Conveyors'], var_name='Metric', value_name='CV_pct')
cv_melted['Metric'] = cv_melted['Metric'].map({
    'StartupTime_s': 'Startup Time',
    'FPS_Mean': 'Mean FPS',
    'FrameTime_Mean_ms': 'Mean Frame Time',
})

plt.figure(figsize=(8, 4.5))
sns.barplot(data=cv_melted, x='Conveyors', y='CV_pct', hue='Metric', palette='Set2')
plt.title("Experimental Coefficient of Variation (CV %)")
plt.xlabel("Number of Conveyors")
plt.ylabel("Coefficient of Variation (%)")
plt.legend()
plt.show()
